# Notebook 07: Modularization and Reusable PySpark Components

## Purpose

This notebook refactors repeated PaySim pipeline logic into reusable Python
modules under `src/paysim_pipeline/`.

The modularized package will contain:

- pipeline configuration;
- Spark session creation;
- explicit schemas;
- Bronze ingestion;
- Silver transformation;
- Gold table construction;
- data-quality validation;
- audit-record generation;
- output utilities.

## Engineering goals

1. Separate business logic from notebook exploration.
2. Avoid copying the same transformations across notebooks.
3. Make individual functions testable.
4. Centralize paths and Spark settings.
5. Prepare the project for pipeline orchestration.
6. Improve maintainability and portfolio presentation.

In [1]:
from pathlib import Path
import os
import sys

from pyspark.sql import functions as F

In [2]:
current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

SRC_PATH = PROJECT_ROOT / "src"
PACKAGE_PATH = SRC_PATH / "paysim_pipeline"

print("Current directory:", current_path)
print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)
print("Package path:", PACKAGE_PATH)

Current directory: C:\Projects\paysim-financial-data-pipeline\notebooks
Project root: C:\Projects\paysim-financial-data-pipeline
Source path: C:\Projects\paysim-financial-data-pipeline\src
Package path: C:\Projects\paysim-financial-data-pipeline\src\paysim_pipeline


In [3]:
PACKAGE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

print("Package directory created:", PACKAGE_PATH)

Package directory created: C:\Projects\paysim-financial-data-pipeline\src\paysim_pipeline


In [4]:
if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print("src available for imports:", str(SRC_PATH) in sys.path)

src available for imports: True


In [5]:
%%writefile ../src/paysim_pipeline/__init__.py
"""Reusable components for the PaySim financial data pipeline."""

__version__ = "0.1.0"

Overwriting ../src/paysim_pipeline/__init__.py


In [6]:
%%writefile ../src/paysim_pipeline/config.py
"""Central configuration for the PaySim pipeline."""

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PipelineConfig:
    """Paths and runtime settings used by the pipeline."""

    project_root: Path
    raw_csv_path: Path

    application_name: str = "PaySimFinancialDataPipeline"
    spark_master: str = "local[4]"
    driver_memory: str = "8g"
    shuffle_partitions: int = 64
    high_value_threshold: float = 200_000.0
    approximate_distinct_rsd: float = 0.05

    @property
    def data_path(self) -> Path:
        return self.project_root / "data"

    @property
    def bronze_path(self) -> Path:
        return self.data_path / "bronze"

    @property
    def silver_path(self) -> Path:
        return self.data_path / "silver"

    @property
    def gold_path(self) -> Path:
        return self.data_path / "gold"

    @property
    def gold_summary_output_path(self) -> Path:
        return self.gold_path / "summary_exports"

    @property
    def audit_output_path(self) -> Path:
        return self.gold_path / "pipeline_audit"

    @property
    def spark_temp_path(self) -> Path:
        return self.project_root / "spark-temp"

    def create_directories(self) -> None:
        """Create pipeline output directories when they do not exist."""

        required_paths = [
            self.bronze_path,
            self.silver_path,
            self.gold_path,
            self.gold_summary_output_path,
            self.audit_output_path,
            self.spark_temp_path,
        ]

        for path in required_paths:
            path.mkdir(
                parents=True,
                exist_ok=True,
            )

    def validate(self) -> None:
        """Validate essential configuration values."""

        if not self.project_root.exists():
            raise FileNotFoundError(
                f"Project root does not exist: {self.project_root}"
            )

        if not self.raw_csv_path.exists():
            raise FileNotFoundError(
                f"Raw CSV does not exist: {self.raw_csv_path}"
            )

        if self.shuffle_partitions <= 0:
            raise ValueError(
                "shuffle_partitions must be greater than zero."
            )

        if not 0 < self.approximate_distinct_rsd <= 0.39:
            raise ValueError(
                "approximate_distinct_rsd must be between 0 and 0.39."
            )

Overwriting ../src/paysim_pipeline/config.py


In [7]:
%%writefile ../src/paysim_pipeline/spark_session.py
"""Spark-session utilities."""

import os
import sys

from pyspark.sql import SparkSession

from paysim_pipeline.config import PipelineConfig


def create_spark_session(
    config: PipelineConfig,
) -> SparkSession:
    """Create and configure a local Spark session."""

    python_executable = sys.executable

    os.environ["PYSPARK_PYTHON"] = python_executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable

    config.spark_temp_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    spark = (
        SparkSession.builder
        .appName(config.application_name)
        .master(config.spark_master)
        .config(
            "spark.driver.memory",
            config.driver_memory,
        )
        .config(
            "spark.sql.shuffle.partitions",
            str(config.shuffle_partitions),
        )
        .config(
            "spark.local.dir",
            str(config.spark_temp_path),
        )
        .config(
            "spark.sql.session.timeZone",
            "UTC",
        )
        .config(
            "spark.driver.host",
            "127.0.0.1",
        )
        .config(
            "spark.driver.bindAddress",
            "127.0.0.1",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

Overwriting ../src/paysim_pipeline/spark_session.py


In [8]:
%%writefile ../src/paysim_pipeline/schemas.py
"""Explicit Spark schemas used by the PaySim pipeline."""

from pyspark.sql import types as T


RAW_TRANSACTION_SCHEMA = T.StructType(
    [
        T.StructField("step", T.IntegerType(), False),
        T.StructField("type", T.StringType(), False),
        T.StructField("amount", T.DoubleType(), False),
        T.StructField("nameOrig", T.StringType(), False),
        T.StructField("oldbalanceOrg", T.DoubleType(), False),
        T.StructField("newbalanceOrig", T.DoubleType(), False),
        T.StructField("nameDest", T.StringType(), False),
        T.StructField("oldbalanceDest", T.DoubleType(), False),
        T.StructField("newbalanceDest", T.DoubleType(), False),
        T.StructField("isFraud", T.IntegerType(), False),
        T.StructField("isFlaggedFraud", T.IntegerType(), False),
    ]
)


BRONZE_REQUIRED_COLUMNS = [
    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "origin_old_balance",
    "origin_new_balance",
    "destination_account",
    "destination_old_balance",
    "destination_new_balance",
    "is_fraud",
    "is_flagged_fraud",
]


SILVER_REQUIRED_COLUMNS = BRONZE_REQUIRED_COLUMNS + [
    "transaction_day",
    "transaction_hour",
    "is_high_value_transaction",
    "origin_balance_error",
    "destination_balance_error",
]

Overwriting ../src/paysim_pipeline/schemas.py


In [9]:
%%writefile ../src/paysim_pipeline/schemas.py
"""Explicit Spark schemas used by the PaySim pipeline."""

from pyspark.sql import types as T


RAW_TRANSACTION_SCHEMA = T.StructType(
    [
        T.StructField("step", T.IntegerType(), False),
        T.StructField("type", T.StringType(), False),
        T.StructField("amount", T.DoubleType(), False),
        T.StructField("nameOrig", T.StringType(), False),
        T.StructField("oldbalanceOrg", T.DoubleType(), False),
        T.StructField("newbalanceOrig", T.DoubleType(), False),
        T.StructField("nameDest", T.StringType(), False),
        T.StructField("oldbalanceDest", T.DoubleType(), False),
        T.StructField("newbalanceDest", T.DoubleType(), False),
        T.StructField("isFraud", T.IntegerType(), False),
        T.StructField("isFlaggedFraud", T.IntegerType(), False),
    ]
)


BRONZE_REQUIRED_COLUMNS = [
    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "origin_old_balance",
    "origin_new_balance",
    "destination_account",
    "destination_old_balance",
    "destination_new_balance",
    "is_fraud",
    "is_flagged_fraud",
]


SILVER_REQUIRED_COLUMNS = BRONZE_REQUIRED_COLUMNS + [
    "transaction_day",
    "transaction_hour",
    "is_high_value_transaction",
    "origin_balance_error",
    "destination_balance_error",
]

Overwriting ../src/paysim_pipeline/schemas.py


In [10]:
%%writefile ../src/paysim_pipeline/validation.py
"""Reusable data-quality validation functions."""

from collections.abc import Sequence

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def find_missing_columns(
    dataframe: DataFrame,
    required_columns: Sequence[str],
) -> list[str]:
    """Return required columns absent from a DataFrame."""

    available_columns = set(dataframe.columns)

    return [
        column
        for column in required_columns
        if column not in available_columns
    ]


def validate_required_columns(
    dataframe: DataFrame,
    required_columns: Sequence[str],
    dataframe_name: str,
) -> None:
    """Raise an error when required columns are absent."""

    missing_columns = find_missing_columns(
        dataframe=dataframe,
        required_columns=required_columns,
    )

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            f"{missing_columns}"
        )


def build_null_profile(
    dataframe: DataFrame,
) -> DataFrame:
    """Return one row containing null counts for all columns."""

    expressions = [
        F.sum(
            F.when(
                F.col(column).isNull(),
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias(column)
        for column in dataframe.columns
    ]

    return dataframe.agg(*expressions)


def count_duplicate_transactions(
    dataframe: DataFrame,
    key_columns: Sequence[str],
) -> int:
    """Count duplicate rows based on a proposed business key."""

    return (
        dataframe
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )


def validate_binary_column(
    dataframe: DataFrame,
    column_name: str,
) -> DataFrame:
    """Return invalid values found in a binary indicator column."""

    return (
        dataframe
        .filter(
            F.col(column_name).isNotNull()
            & ~F.col(column_name).isin(0, 1)
        )
        .select(column_name)
        .distinct()
    )


def reconciliation_result(
    source_count: int,
    target_count: int,
) -> dict:
    """Build a reusable row-count reconciliation result."""

    return {
        "source_row_count": source_count,
        "target_row_count": target_count,
        "row_count_difference": target_count - source_count,
        "reconciliation_status": (
            "PASS"
            if source_count == target_count
            else "FAIL"
        ),
    }

Overwriting ../src/paysim_pipeline/validation.py


In [11]:
%%writefile ../src/paysim_pipeline/bronze.py
"""Bronze-layer ingestion functions."""

from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from paysim_pipeline.schemas import RAW_TRANSACTION_SCHEMA


RAW_TO_CANONICAL_COLUMN_MAP = {
    "type": "transaction_type",
    "nameOrig": "origin_account",
    "oldbalanceOrg": "origin_old_balance",
    "newbalanceOrig": "origin_new_balance",
    "nameDest": "destination_account",
    "oldbalanceDest": "destination_old_balance",
    "newbalanceDest": "destination_new_balance",
    "isFraud": "is_fraud",
    "isFlaggedFraud": "is_flagged_fraud",
}


def read_raw_transactions(
    spark: SparkSession,
    raw_csv_path: Path,
) -> DataFrame:
    """Read the PaySim CSV using an explicit schema."""

    return (
        spark.read
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(RAW_TRANSACTION_SCHEMA)
        .csv(str(raw_csv_path))
    )


def standardize_bronze_columns(
    raw_dataframe: DataFrame,
) -> DataFrame:
    """Rename raw PaySim columns to canonical pipeline names."""

    bronze_dataframe = raw_dataframe

    for source_column, target_column in (
        RAW_TO_CANONICAL_COLUMN_MAP.items()
    ):
        bronze_dataframe = bronze_dataframe.withColumnRenamed(
            source_column,
            target_column,
        )

    return bronze_dataframe


def add_bronze_metadata(
    dataframe: DataFrame,
    source_file_name: str,
    pipeline_run_id: str,
) -> DataFrame:
    """Add ingestion metadata to Bronze records."""

    return (
        dataframe
        .withColumn(
            "source_file_name",
            F.lit(source_file_name),
        )
        .withColumn(
            "pipeline_run_id",
            F.lit(pipeline_run_id),
        )
        .withColumn(
            "bronze_ingestion_timestamp",
            F.current_timestamp(),
        )
    )


def build_bronze_transactions(
    spark: SparkSession,
    raw_csv_path: Path,
    pipeline_run_id: str,
) -> DataFrame:
    """Run the complete Bronze ingestion transformation."""

    raw_dataframe = read_raw_transactions(
        spark=spark,
        raw_csv_path=raw_csv_path,
    )

    standardized_dataframe = standardize_bronze_columns(
        raw_dataframe=raw_dataframe,
    )

    return add_bronze_metadata(
        dataframe=standardized_dataframe,
        source_file_name=raw_csv_path.name,
        pipeline_run_id=pipeline_run_id,
    )

Overwriting ../src/paysim_pipeline/bronze.py


In [12]:
%%writefile ../src/paysim_pipeline/silver.py
"""Silver-layer cleansing and enrichment functions."""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from paysim_pipeline.schemas import BRONZE_REQUIRED_COLUMNS
from paysim_pipeline.validation import validate_required_columns


VALID_TRANSACTION_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]


def select_valid_transactions(
    bronze_dataframe: DataFrame,
) -> DataFrame:
    """Apply the core validity rules for PaySim transactions."""

    validate_required_columns(
        dataframe=bronze_dataframe,
        required_columns=BRONZE_REQUIRED_COLUMNS,
        dataframe_name="bronze_dataframe",
    )

    return bronze_dataframe.filter(
        F.col("step").isNotNull()
        & (F.col("step") > 0)
        & F.col("transaction_type").isin(
            *VALID_TRANSACTION_TYPES
        )
        & F.col("amount").isNotNull()
        & (F.col("amount") >= 0)
        & F.col("origin_account").isNotNull()
        & F.col("destination_account").isNotNull()
        & F.col("is_fraud").isin(0, 1)
        & F.col("is_flagged_fraud").isin(0, 1)
    )


def add_time_features(
    dataframe: DataFrame,
) -> DataFrame:
    """Derive day and hour features from the PaySim step."""

    return (
        dataframe
        .withColumn(
            "transaction_day",
            (
                F.floor(
                    (F.col("step") - F.lit(1))
                    / F.lit(24)
                )
                + F.lit(1)
            ).cast("integer"),
        )
        .withColumn(
            "transaction_hour",
            F.pmod(
                F.col("step") - F.lit(1),
                F.lit(24),
            ).cast("integer"),
        )
    )


def add_balance_features(
    dataframe: DataFrame,
) -> DataFrame:
    """Create balance-difference and balance-error fields."""

    return (
        dataframe
        .withColumn(
            "origin_balance_change",
            F.round(
                F.col("origin_old_balance")
                - F.col("origin_new_balance"),
                2,
            ),
        )
        .withColumn(
            "destination_balance_change",
            F.round(
                F.col("destination_new_balance")
                - F.col("destination_old_balance"),
                2,
            ),
        )
        .withColumn(
            "origin_balance_error",
            F.round(
                F.col("origin_old_balance")
                - F.col("amount")
                - F.col("origin_new_balance"),
                2,
            ),
        )
        .withColumn(
            "destination_balance_error",
            F.round(
                F.col("destination_old_balance")
                + F.col("amount")
                - F.col("destination_new_balance"),
                2,
            ),
        )
    )


def add_transaction_indicators(
    dataframe: DataFrame,
    high_value_threshold: float,
) -> DataFrame:
    """Add reusable monitoring and segmentation indicators."""

    return (
        dataframe
        .withColumn(
            "is_high_value_transaction",
            F.when(
                F.col("amount")
                >= F.lit(high_value_threshold),
                F.lit(1),
            ).otherwise(F.lit(0)),
        )
        .withColumn(
            "is_zero_amount_transaction",
            F.when(
                F.col("amount") == 0,
                F.lit(1),
            ).otherwise(F.lit(0)),
        )
        .withColumn(
            "is_origin_merchant",
            F.when(
                F.col("origin_account").startswith("M"),
                F.lit(1),
            ).otherwise(F.lit(0)),
        )
        .withColumn(
            "is_destination_merchant",
            F.when(
                F.col("destination_account").startswith("M"),
                F.lit(1),
            ).otherwise(F.lit(0)),
        )
    )


def build_silver_transactions(
    bronze_dataframe: DataFrame,
    high_value_threshold: float,
) -> DataFrame:
    """Run the complete Silver transformation."""

    valid_dataframe = select_valid_transactions(
        bronze_dataframe=bronze_dataframe,
    )

    time_enriched_dataframe = add_time_features(
        dataframe=valid_dataframe,
    )

    balance_enriched_dataframe = add_balance_features(
        dataframe=time_enriched_dataframe,
    )

    return add_transaction_indicators(
        dataframe=balance_enriched_dataframe,
        high_value_threshold=high_value_threshold,
    )

Overwriting ../src/paysim_pipeline/silver.py


In [13]:
%%writefile ../src/paysim_pipeline/gold.py
"""Gold analytical-table construction functions."""

from pyspark.sql import DataFrame
from pyspark.sql import Window
from pyspark.sql import functions as F

from paysim_pipeline.schemas import SILVER_REQUIRED_COLUMNS
from paysim_pipeline.validation import validate_required_columns


def _safe_percentage(
    numerator_column: str,
    denominator_column: str,
    precision: int = 6,
):
    """Return a divide-by-zero-safe percentage expression."""

    return F.when(
        F.col(denominator_column) > 0,
        F.round(
            F.col(numerator_column)
            / F.col(denominator_column)
            * F.lit(100.0),
            precision,
        ),
    ).otherwise(F.lit(0.0))


def create_daily_transaction_summary(
    silver_dataframe: DataFrame,
    approximate_distinct_rsd: float = 0.05,
) -> DataFrame:
    """Create one row per transaction day."""

    basic_summary = (
        silver_dataframe
        .groupBy("transaction_day")
        .agg(
            F.count("*").alias("transaction_count"),
            F.round(F.sum("amount"), 2).alias(
                "total_transaction_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "average_transaction_amount"
            ),
            F.round(F.min("amount"), 2).alias(
                "minimum_transaction_amount"
            ),
            F.round(F.max("amount"), 2).alias(
                "maximum_transaction_amount"
            ),
            F.sum("is_fraud").alias("fraud_count"),
            F.round(
                F.sum(
                    F.when(
                        F.col("is_fraud") == 1,
                        F.col("amount"),
                    ).otherwise(F.lit(0.0))
                ),
                2,
            ).alias("fraud_amount"),
            F.sum("is_flagged_fraud").alias(
                "flagged_fraud_count"
            ),
            F.sum("is_high_value_transaction").alias(
                "high_value_transaction_count"
            ),
        )
    )

    account_summary = (
        silver_dataframe
        .groupBy("transaction_day")
        .agg(
            F.approx_count_distinct(
                "origin_account",
                approximate_distinct_rsd,
            ).alias(
                "estimated_unique_origin_accounts"
            ),
            F.approx_count_distinct(
                "destination_account",
                approximate_distinct_rsd,
            ).alias(
                "estimated_unique_destination_accounts"
            ),
        )
    )

    return (
        basic_summary
        .join(
            account_summary,
            on="transaction_day",
            how="left",
        )
        .withColumn(
            "fraud_rate_pct",
            _safe_percentage(
                "fraud_count",
                "transaction_count",
            ),
        )
        .withColumn(
            "fraud_amount_pct",
            _safe_percentage(
                "fraud_amount",
                "total_transaction_amount",
            ),
        )
        .orderBy("transaction_day")
    )


def create_daily_type_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per day and transaction type."""

    return (
        silver_dataframe
        .groupBy(
            "transaction_day",
            "transaction_type",
        )
        .agg(
            F.count("*").alias("transaction_count"),
            F.round(F.sum("amount"), 2).alias(
                "total_transaction_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "average_transaction_amount"
            ),
            F.sum("is_fraud").alias("fraud_count"),
            F.round(
                F.sum(
                    F.when(
                        F.col("is_fraud") == 1,
                        F.col("amount"),
                    ).otherwise(F.lit(0.0))
                ),
                2,
            ).alias("fraud_amount"),
        )
        .withColumn(
            "fraud_rate_pct",
            _safe_percentage(
                "fraud_count",
                "transaction_count",
            ),
        )
        .orderBy(
            "transaction_day",
            "transaction_type",
        )
    )


def create_hourly_fraud_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per PaySim hourly step."""

    return (
        silver_dataframe
        .groupBy(
            F.col("step").alias("hourly_step")
        )
        .agg(
            F.count("*").alias("transaction_count"),
            F.sum("is_fraud").alias("fraud_count"),
            F.round(F.sum("amount"), 2).alias(
                "total_transaction_amount"
            ),
            F.round(
                F.sum(
                    F.when(
                        F.col("is_fraud") == 1,
                        F.col("amount"),
                    ).otherwise(F.lit(0.0))
                ),
                2,
            ).alias("fraud_amount"),
        )
        .withColumn(
            "fraud_rate_pct",
            _safe_percentage(
                "fraud_count",
                "transaction_count",
            ),
        )
        .orderBy("hourly_step")
    )


def create_transaction_type_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per transaction type."""

    return (
        silver_dataframe
        .groupBy("transaction_type")
        .agg(
            F.count("*").alias("transaction_count"),
            F.round(F.sum("amount"), 2).alias(
                "total_transaction_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "average_transaction_amount"
            ),
            F.sum("is_fraud").alias("fraud_count"),
            F.sum("is_flagged_fraud").alias(
                "flagged_fraud_count"
            ),
            F.sum("is_high_value_transaction").alias(
                "high_value_transaction_count"
            ),
        )
        .withColumn(
            "fraud_rate_pct",
            _safe_percentage(
                "fraud_count",
                "transaction_count",
            ),
        )
        .orderBy("transaction_type")
    )


def create_origin_account_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per origin account."""

    return (
        silver_dataframe
        .groupBy("origin_account")
        .agg(
            F.count("*").alias(
                "origin_transaction_count"
            ),
            F.round(F.sum("amount"), 2).alias(
                "origin_total_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "origin_average_amount"
            ),
            F.sum("is_fraud").alias(
                "origin_fraud_count"
            ),
            F.max("step").alias(
                "latest_origin_transaction_step"
            ),
        )
    )


def create_destination_account_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per destination account."""

    return (
        silver_dataframe
        .groupBy("destination_account")
        .agg(
            F.count("*").alias(
                "destination_transaction_count"
            ),
            F.round(F.sum("amount"), 2).alias(
                "destination_total_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "destination_average_amount"
            ),
            F.sum("is_fraud").alias(
                "destination_fraud_count"
            ),
            F.max("step").alias(
                "latest_destination_transaction_step"
            ),
        )
    )


def create_high_value_summary(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create one row per day and type for high-value transactions."""

    return (
        silver_dataframe
        .filter(
            F.col("is_high_value_transaction") == 1
        )
        .groupBy(
            "transaction_day",
            "transaction_type",
        )
        .agg(
            F.count("*").alias(
                "high_value_transaction_count"
            ),
            F.round(F.sum("amount"), 2).alias(
                "high_value_total_amount"
            ),
            F.round(F.avg("amount"), 2).alias(
                "high_value_average_amount"
            ),
            F.sum("is_fraud").alias(
                "high_value_fraud_count"
            ),
        )
        .orderBy(
            "transaction_day",
            "transaction_type",
        )
    )


def create_fraud_monitoring_table(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Return transaction-level fraudulent records."""

    return (
        silver_dataframe
        .filter(F.col("is_fraud") == 1)
        .select(
            "step",
            "transaction_day",
            "transaction_hour",
            "transaction_type",
            "amount",
            "origin_account",
            "destination_account",
            "is_fraud",
            "is_flagged_fraud",
            "is_high_value_transaction",
            "pipeline_run_id",
        )
        .orderBy(
            "step",
            "origin_account",
        )
    )


def create_fraud_feature_table(
    silver_dataframe: DataFrame,
) -> DataFrame:
    """Create leakage-aware transaction features."""

    origin_history_window = (
        Window
        .partitionBy("origin_account")
        .orderBy("step")
        .rowsBetween(
            Window.unboundedPreceding,
            -1,
        )
    )

    return (
        silver_dataframe
        .withColumn(
            "prior_origin_transaction_count",
            F.count("*").over(
                origin_history_window
            ),
        )
        .withColumn(
            "prior_origin_average_amount",
            F.round(
                F.avg("amount").over(
                    origin_history_window
                ),
                2,
            ),
        )
        .withColumn(
            "prior_origin_maximum_amount",
            F.round(
                F.max("amount").over(
                    origin_history_window
                ),
                2,
            ),
        )
        .select(
            "step",
            "transaction_day",
            "transaction_hour",
            "transaction_type",
            "amount",
            "origin_account",
            "destination_account",
            "is_high_value_transaction",
            "is_destination_merchant",
            "prior_origin_transaction_count",
            "prior_origin_average_amount",
            "prior_origin_maximum_amount",
            "is_fraud",
        )
    )


def build_gold_tables(
    silver_dataframe: DataFrame,
    approximate_distinct_rsd: float = 0.05,
) -> dict[str, DataFrame]:
    """Create and return all Gold tables."""

    validate_required_columns(
        dataframe=silver_dataframe,
        required_columns=SILVER_REQUIRED_COLUMNS,
        dataframe_name="silver_dataframe",
    )

    return {
        "daily_transaction_summary": (
            create_daily_transaction_summary(
                silver_dataframe,
                approximate_distinct_rsd,
            )
        ),
        "daily_type_summary": (
            create_daily_type_summary(
                silver_dataframe
            )
        ),
        "hourly_fraud_summary": (
            create_hourly_fraud_summary(
                silver_dataframe
            )
        ),
        "transaction_type_summary": (
            create_transaction_type_summary(
                silver_dataframe
            )
        ),
        "origin_account_summary": (
            create_origin_account_summary(
                silver_dataframe
            )
        ),
        "destination_account_summary": (
            create_destination_account_summary(
                silver_dataframe
            )
        ),
        "high_value_summary": (
            create_high_value_summary(
                silver_dataframe
            )
        ),
        "fraud_monitoring": (
            create_fraud_monitoring_table(
                silver_dataframe
            )
        ),
        "fraud_feature": (
            create_fraud_feature_table(
                silver_dataframe
            )
        ),
    }

Overwriting ../src/paysim_pipeline/gold.py


In [14]:
%%writefile ../src/paysim_pipeline/audit.py
"""Pipeline audit-record utilities."""

from datetime import datetime, timezone

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import types as T


AUDIT_SCHEMA = T.StructType(
    [
        T.StructField(
            "pipeline_run_id",
            T.StringType(),
            False,
        ),
        T.StructField(
            "pipeline_stage",
            T.StringType(),
            False,
        ),
        T.StructField(
            "table_name",
            T.StringType(),
            False,
        ),
        T.StructField(
            "row_count",
            T.LongType(),
            False,
        ),
        T.StructField(
            "column_count",
            T.IntegerType(),
            False,
        ),
        T.StructField(
            "reconciliation_status",
            T.StringType(),
            False,
        ),
        T.StructField(
            "execution_time_seconds",
            T.DoubleType(),
            False,
        ),
        T.StructField(
            "audit_timestamp_utc",
            T.TimestampType(),
            False,
        ),
    ]
)


def create_audit_record(
    spark: SparkSession,
    dataframe: DataFrame,
    pipeline_run_id: str,
    pipeline_stage: str,
    table_name: str,
    execution_time_seconds: float,
    reconciliation_status: str = "PASS",
) -> DataFrame:
    """Create a one-row Spark audit DataFrame."""

    audit_row = [
        (
            pipeline_run_id,
            pipeline_stage,
            table_name,
            dataframe.count(),
            len(dataframe.columns),
            reconciliation_status,
            float(execution_time_seconds),
            datetime.now(timezone.utc).replace(
                tzinfo=None
            ),
        )
    ]

    return spark.createDataFrame(
        audit_row,
        schema=AUDIT_SCHEMA,
    )


def combine_audit_records(
    audit_dataframes: list[DataFrame],
) -> DataFrame:
    """Union multiple audit DataFrames."""

    if not audit_dataframes:
        raise ValueError(
            "At least one audit DataFrame is required."
        )

    combined_dataframe = audit_dataframes[0]

    for audit_dataframe in audit_dataframes[1:]:
        combined_dataframe = (
            combined_dataframe.unionByName(
                audit_dataframe
            )
        )

    return combined_dataframe

Overwriting ../src/paysim_pipeline/audit.py


In [15]:
%%writefile ../src/paysim_pipeline/io_utils.py
"""Reusable pipeline output utilities."""

from pathlib import Path

from pyspark.sql import DataFrame


def export_small_dataframe_to_csv(
    dataframe: DataFrame,
    output_path: Path,
    file_name: str,
    index: bool = False,
) -> Path:
    """Export a small Spark DataFrame as one local CSV file."""

    output_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_path = output_path / file_name

    pandas_dataframe = dataframe.toPandas()

    pandas_dataframe.to_csv(
        final_path,
        index=index,
    )

    return final_path


def write_dataframe_to_parquet(
    dataframe: DataFrame,
    output_path: Path,
    mode: str = "overwrite",
    partition_columns: list[str] | None = None,
) -> None:
    """Write a large Spark DataFrame to Parquet."""

    writer = dataframe.write.mode(mode)

    if partition_columns:
        writer = writer.partitionBy(
            *partition_columns
        )

    writer.parquet(
        str(output_path)
    )


def clear_csv_outputs(
    output_path: Path,
) -> list[Path]:
    """Delete existing CSV outputs and return their paths."""

    output_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    deleted_files = []

    for file_path in output_path.glob("*.csv"):
        file_path.unlink()
        deleted_files.append(file_path)

    return deleted_files

Overwriting ../src/paysim_pipeline/io_utils.py


Part B - Import and test the package

In [16]:
%load_ext autoreload
%autoreload 2

In [17]:
%reload_ext autoreload

In [18]:
from paysim_pipeline.config import PipelineConfig
from paysim_pipeline.spark_session import create_spark_session

from paysim_pipeline.bronze import (
    build_bronze_transactions,
)

from paysim_pipeline.silver import (
    build_silver_transactions,
)

from paysim_pipeline.gold import (
    build_gold_tables,
    create_daily_transaction_summary,
    create_transaction_type_summary,
)

from paysim_pipeline.validation import (
    build_null_profile,
    validate_required_columns,
)

from paysim_pipeline.schemas import (
    BRONZE_REQUIRED_COLUMNS,
    SILVER_REQUIRED_COLUMNS,
)

from paysim_pipeline.io_utils import (
    clear_csv_outputs,
    export_small_dataframe_to_csv,
)

In [19]:
RAW_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

config = PipelineConfig(
    project_root=PROJECT_ROOT,
    raw_csv_path=RAW_CSV_PATH,
    application_name="PaySimModularization",
    spark_master="local[4]",
    driver_memory="8g",
    shuffle_partitions=64,
    high_value_threshold=200_000.0,
    approximate_distinct_rsd=0.05,
)

config.create_directories()
config.validate()

print("Configuration validated.")
print("Raw file:", config.raw_csv_path)
print("Gold path:", config.gold_path)
print("Driver memory:", config.driver_memory)
print("Shuffle partitions:", config.shuffle_partitions)

Configuration validated.
Raw file: C:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
Gold path: C:\Projects\paysim-financial-data-pipeline\data\gold
Driver memory: 8g
Shuffle partitions: 64


In [20]:
from datetime import datetime, timezone
import uuid

run_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

pipeline_run_id = (
    f"{run_timestamp}_{uuid.uuid4().hex[:8]}"
)

print("Pipeline run ID:", pipeline_run_id)

Pipeline run ID: 20260726_002737_5b7c982a


In [21]:
from datetime import datetime, timezone
import uuid

run_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

pipeline_run_id = (
    f"{run_timestamp}_{uuid.uuid4().hex[:8]}"
)

print("Pipeline run ID:", pipeline_run_id)

Pipeline run ID: 20260726_002738_a2366ff7


In [22]:
spark = create_spark_session(
    config=config,
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions"),
)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark master: local[4]
Shuffle partitions: 64


In [23]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [24]:
bronze_df = build_bronze_transactions(
    spark=spark,
    raw_csv_path=config.raw_csv_path,
    pipeline_run_id=pipeline_run_id,
)

bronze_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- origin_account: string (nullable = true)
 |-- origin_old_balance: double (nullable = true)
 |-- origin_new_balance: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- destination_old_balance: double (nullable = true)
 |-- destination_new_balance: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- is_flagged_fraud: integer (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- pipeline_run_id: string (nullable = false)
 |-- bronze_ingestion_timestamp: timestamp (nullable = false)



In [25]:
bronze_sample_df = bronze_df.limit(1000)

bronze_sample_df.show(
    n=5,
    truncate=False,
)

+----+----------------+--------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+------------------------------------+------------------------+--------------------------+
|step|transaction_type|amount  |origin_account|origin_old_balance|origin_new_balance|destination_account|destination_old_balance|destination_new_balance|is_fraud|is_flagged_fraud|source_file_name                    |pipeline_run_id         |bronze_ingestion_timestamp|
+----+----------------+--------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+------------------------------------+------------------------+--------------------------+
|1   |PAYMENT         |9839.64 |C1231006815   |170136.0          |160296.36         |M1979787155        |0.0                    |0.0                    |0       |0               |PS_20174392719

In [26]:
validate_required_columns(
    dataframe=bronze_df,
    required_columns=BRONZE_REQUIRED_COLUMNS,
    dataframe_name="bronze_df",
)

print("Bronze required-column validation: PASS")

Bronze required-column validation: PASS


In [27]:
silver_df = build_silver_transactions(
    bronze_dataframe=bronze_df,
    high_value_threshold=(
        config.high_value_threshold
    ),
)

silver_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- origin_account: string (nullable = true)
 |-- origin_old_balance: double (nullable = true)
 |-- origin_new_balance: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- destination_old_balance: double (nullable = true)
 |-- destination_new_balance: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- is_flagged_fraud: integer (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- pipeline_run_id: string (nullable = false)
 |-- bronze_ingestion_timestamp: timestamp (nullable = false)
 |-- transaction_day: integer (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- origin_balance_change: double (nullable = true)
 |-- destination_balance_change: double (nullable = true)
 |-- origin_balance_error: double (nullable = true)
 |-- destination_balance_error: double (nullable = true)

In [28]:
validate_required_columns(
    dataframe=silver_df,
    required_columns=SILVER_REQUIRED_COLUMNS,
    dataframe_name="silver_df",
)

print("Silver required-column validation: PASS")

Silver required-column validation: PASS


In [29]:
silver_df.select(
    "step",
    "transaction_day",
    "transaction_hour",
    "transaction_type",
    "amount",
    "origin_account",
    "destination_account",
    "is_fraud",
    "is_high_value_transaction",
).show(
    n=10,
    truncate=False,
)

+----+---------------+----------------+----------------+--------+--------------+-------------------+--------+-------------------------+
|step|transaction_day|transaction_hour|transaction_type|amount  |origin_account|destination_account|is_fraud|is_high_value_transaction|
+----+---------------+----------------+----------------+--------+--------------+-------------------+--------+-------------------------+
|1   |1              |0               |PAYMENT         |9839.64 |C1231006815   |M1979787155        |0       |0                        |
|1   |1              |0               |PAYMENT         |1864.28 |C1666544295   |M2044282225        |0       |0                        |
|1   |1              |0               |TRANSFER        |181.0   |C1305486145   |C553264065         |1       |0                        |
|1   |1              |0               |CASH_OUT        |181.0   |C840083671    |C38997010          |1       |0                        |
|1   |1              |0               |PAYMENT  

In [30]:
silver_null_profile_df = build_null_profile(
    silver_df.limit(100_000)
)

silver_null_profile_df.show(
    truncate=False,
)

+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+----------------+---------------+--------------------------+---------------+----------------+---------------------+--------------------------+--------------------+-------------------------+-------------------------+--------------------------+------------------+-----------------------+
|step|transaction_type|amount|origin_account|origin_old_balance|origin_new_balance|destination_account|destination_old_balance|destination_new_balance|is_fraud|is_flagged_fraud|source_file_name|pipeline_run_id|bronze_ingestion_timestamp|transaction_day|transaction_hour|origin_balance_change|destination_balance_change|origin_balance_error|destination_balance_error|is_high_value_transaction|is_zero_amount_transaction|is_origin_merchant|is_destination_merchant|
+----+----------------+------+--------------+------------------+----------

In [31]:
test_transaction_type_summary_df = (
    create_transaction_type_summary(
        silver_dataframe=silver_df,
    )
)

test_transaction_type_summary_df.show(
    truncate=False,
)

+----------------+-----------------+------------------------+--------------------------+-----------+-------------------+----------------------------+--------------+
|transaction_type|transaction_count|total_transaction_amount|average_transaction_amount|fraud_count|flagged_fraud_count|high_value_transaction_count|fraud_rate_pct|
+----------------+-----------------+------------------------+--------------------------+-----------+-------------------+----------------------------+--------------+
|CASH_IN         |1399284          |2.3636739191246E11      |168920.24                 |0          |0                  |475868                      |0.0           |
|CASH_OUT        |2237500          |3.9441299522449E11      |176273.96                 |4116       |0                  |788559                      |0.183955      |
|DEBIT           |41432            |2.2719922128E8          |5483.67                   |0          |0                  |27                          |0.0           |
|PAYMENT  

In [32]:
test_daily_summary_df = (
    create_daily_transaction_summary(
        silver_dataframe=silver_df,
        approximate_distinct_rsd=(
            config.approximate_distinct_rsd
        ),
    )
)

test_daily_summary_df.show(
    n=10,
    truncate=False,
)

+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------------------------+-------------------------------------+--------------+----------------+
|transaction_day|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|fraud_count|fraud_amount  |flagged_fraud_count|high_value_transaction_count|estimated_unique_origin_accounts|estimated_unique_destination_accounts|fraud_rate_pct|fraud_amount_pct|
+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------------------------+-------------------------------------+--------------+----------------+
|1              |574255           |9.

In [33]:
gold_tables = build_gold_tables(
    silver_dataframe=silver_df,
    approximate_distinct_rsd=(
        config.approximate_distinct_rsd
    ),
)

sorted(gold_tables.keys())

['daily_transaction_summary',
 'daily_type_summary',
 'destination_account_summary',
 'fraud_feature',
 'fraud_monitoring',
 'high_value_summary',
 'hourly_fraud_summary',
 'origin_account_summary',
 'transaction_type_summary']

In [34]:
for table_name, dataframe in gold_tables.items():
    print(
        f"{table_name}: "
        f"{len(dataframe.columns)} columns"
    )

daily_transaction_summary: 14 columns
daily_type_summary: 8 columns
hourly_fraud_summary: 6 columns
transaction_type_summary: 8 columns
origin_account_summary: 6 columns
destination_account_summary: 6 columns
high_value_summary: 6 columns
fraud_monitoring: 11 columns
fraud_feature: 13 columns


In [35]:
exported_path = export_small_dataframe_to_csv(
    dataframe=(
        gold_tables[
            "transaction_type_summary"
        ]
    ),
    output_path=(
        config.gold_summary_output_path
    ),
    file_name=(
        "modular_transaction_type_summary_"
        f"{pipeline_run_id}.csv"
    ),
)

print("Exported:", exported_path)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Exported: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\modular_transaction_type_summary_20260726_002738_a2366ff7.csv


In [36]:
print("File exists:", exported_path.exists())
print(
    "File size in bytes:",
    exported_path.stat().st_size,
)

File exists: True
File size in bytes: 451


In [37]:
# deleted_files = clear_csv_outputs(
#     config.gold_summary_output_path
# )

# for deleted_file in deleted_files:
#     print("Deleted:", deleted_file.name)

In [38]:
help(create_daily_transaction_summary)

Help on function create_daily_transaction_summary in module paysim_pipeline.gold:

create_daily_transaction_summary(silver_dataframe: pyspark.sql.dataframe.DataFrame, approximate_distinct_rsd: float = 0.05) -> pyspark.sql.dataframe.DataFrame
    Create one row per transaction day.



In [39]:
help(create_daily_transaction_summary)

Help on function create_daily_transaction_summary in module paysim_pipeline.gold:

create_daily_transaction_summary(silver_dataframe: pyspark.sql.dataframe.DataFrame, approximate_distinct_rsd: float = 0.05) -> pyspark.sql.dataframe.DataFrame
    Create one row per transaction day.



## Notebook-to-module mapping

| Previous notebook responsibility | Reusable module |
|---|---|
| Path and runtime variables | `config.py` |
| Spark initialization | `spark_session.py` |
| Explicit schemas | `schemas.py` |
| Raw CSV ingestion | `bronze.py` |
| Cleansing and enrichment | `silver.py` |
| Analytical tables | `gold.py` |
| Data-quality checks | `validation.py` |
| Pipeline audit records | `audit.py` |
| CSV and Parquet output | `io_utils.py` |

The notebooks now become development and demonstration interfaces. Core
pipeline behavior resides in importable Python modules.

## Engineering decisions

1. Configuration is centralized in an immutable dataclass.

2. Spark creation is isolated from business transformations.

3. An explicit raw schema prevents unreliable type inference.

4. Bronze, Silver, and Gold logic is separated by responsibility.

5. Transformation functions accept DataFrames and return DataFrames.

6. Most functions do not call actions such as `count()`, `show()`, or
   `collect()`.

7. Gold tables are returned in a dictionary for orchestration.

8. Exact high-cardinality account counts are avoided in daily monitoring.

9. Small summary tables may be exported through Pandas.

10. Large account and transaction-level outputs remain in Spark.

11. Output cleanup is explicit and configurable.

12. Audit logic is separated from transformation logic.

13. Fraud-feature history uses only prior origin-account transactions.

14. Raw balance columns are excluded from the fraud feature table to reduce
    documented target-leakage risk.

15. Notebook 07 tests individual components rather than executing the entire
    production pipeline.

In [40]:
sorted(
    path.name
    for path in PACKAGE_PATH.glob("*.py")
)

['__init__.py',
 'audit.py',
 'bronze.py',
 'config.py',
 'gold.py',
 'io_utils.py',
 'schemas.py',
 'silver.py',
 'spark_session.py',
 'validation.py']

In [41]:
import paysim_pipeline

print(
    "PaySim package version:",
    paysim_pipeline.__version__,
)

PaySim package version: 0.1.0


In [42]:
for dataframe_name in [
    "bronze_df",
    "silver_df",
]:
    dataframe = globals().get(dataframe_name)

    if dataframe is not None:
        dataframe.unpersist(
            blocking=False
        )

spark.catalog.clearCache()

print("Spark cache cleared.")

Spark cache cleared.


In [43]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
